In [6]:
import os
import numpy as np
import pandas as pd
from xgboost import DMatrix, train
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings; warnings.filterwarnings("ignore")

CSV_PATH = "/home/diya/Downloads/cv/merged_ml_dataset.csv"
assert os.path.exists(CSV_PATH), f"CSV not found at: {CSV_PATH}"

df = pd.read_csv(CSV_PATH)
assert "label" in df.columns, "Expected a 'label' column in the CSV."
feature_cols = [c for c in df.columns if c not in ["label", "filename"]]
X = df[feature_cols]
y = df["label"].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

classes = np.unique(y_train)
if set(classes) == {0, 1}:
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    scale_pos_weight = (neg / pos) if pos > 0 else 1.0
else:
    scale_pos_weight = 1.0

dtrain = DMatrix(X_train, label=y_train)
dvalid = DMatrix(X_valid, label=y_valid)

params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1.0,
    "gamma": 0.0,
    "lambda": 1.0,
    "scale_pos_weight": scale_pos_weight,
    "tree_method": "hist",
    "seed": 42
}

watchlist = [(dtrain, "train"), (dvalid, "eval")]

model = train(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,
    evals=watchlist,
    early_stopping_rounds=50,
    verbose_eval=False
)

y_prob = model.predict(DMatrix(X_valid))
y_pred = (y_prob >= 0.5).astype(int)

print("\nClassification report (validation):")
print(classification_report(y_valid, y_pred, digits=4))

auc = roc_auc_score(y_valid, y_prob)
print(f"AUC: {auc:.4f}")

cm = confusion_matrix(y_valid, y_pred, labels=[0,1])
print("\nConfusion Matrix [rows=true, cols=pred]:\n", cm)



Classification report (validation):
              precision    recall  f1-score   support

           0     0.8505    0.8381    0.8443      5968
           1     0.8675    0.8780    0.8727      7204

    accuracy                         0.8599     13172
   macro avg     0.8590    0.8581    0.8585     13172
weighted avg     0.8598    0.8599    0.8598     13172

AUC: 0.9389

Confusion Matrix [rows=true, cols=pred]:
 [[5002  966]
 [ 879 6325]]
